# Train the vaccine sentiment model (Colab)

Fine-tunes `answerdotai/ModernBERT-base` on `train3.csv` / `test3.csv` and saves the result to Google Drive,
ready to drop into `backend/model_sentiment/` for the model service.

**Before running**
1. Runtime → Change runtime type → **T4 GPU**.
2. In Google Drive, create `MyDrive/vax-sentiment/` and upload `train3.csv` and `test3.csv` from the repo's `data/sentiment/` into it.
3. Runtime → Run all. Training takes about 10–15 minutes.

**Afterwards**, download everything in `MyDrive/vax-sentiment/modernbert_sentiment_v3/` into
`backend/model_sentiment/modernbert_sentiment_v3/` in the repo (not committed to git).

In [ ]:
!pip install -q "transformers>=4.48" datasets

In [ ]:
import torch
from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/vax-sentiment"
SAVE_DIR = f"{DATA_DIR}/modernbert_sentiment_v3"   # final model, kept in Drive
CHECKPOINT_DIR = "/content/checkpoints"             # temporary, wiped with the runtime

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - switch the runtime to T4 GPU")

In [ ]:
import re
import pandas as pd

LABELS = {0: "negative", 1: "neutral", 2: "positive"}

# Keep in sync with clean_text() in backend/model_sentiment/app.py so the model
# sees the same kind of text at training and prediction time.
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticons
    "\U0001F300-\U0001F5FF"  # symbols & pictographs
    "\U0001F680-\U0001F6FF"  # transport & map symbols
    "\U0001F1E0-\U0001F1FF"  # flags
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251"
    "\U0001F900-\U0001F9FF"  # supplemental symbols
    "\U00002600-\U000026FF"  # misc symbols
    "]+",
    flags=re.UNICODE,
)

def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+", "", text)                          # remove links
    text = re.sub(r"<(user|url)>", "", text, flags=re.IGNORECASE) # remove dataset placeholders
    text = re.sub(r"@\w+", "", text)                             # remove @mentions
    text = re.sub(r"#(\w+)", r"\1", text)                        # #vaccine -> vaccine
    text = text.replace('"', "").strip()
    text = re.sub(r"\s+", " ", text)                             # collapse extra spaces
    return EMOJI_PATTERN.sub("", text)

train_df = pd.read_csv(f"{DATA_DIR}/train3.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test3.csv")
for df in (train_df, test_df):
    df["text"] = df["safe_text"].map(clean_text)
train_df = train_df[train_df["text"].str.len() > 0]
test_df = test_df[test_df["text"].str.len() > 0]

print(len(train_df), "train rows,", len(test_df), "test rows")
print(train_df["label"].map(LABELS).value_counts(normalize=True).round(3))

In [ ]:
import numpy as np
import torch.nn as nn
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=LABELS,
    label2id={v: k for k, v in LABELS.items()},
)

def to_dataset(df):
    ds = Dataset.from_pandas(df[["text", "label"]].reset_index(drop=True)).rename_column("label", "labels")
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=256), batched=True).remove_columns(["text"])

train_ds, test_ds = to_dataset(train_df), to_dataset(test_df)

# Negative is the smallest class, so it gets a bigger weight in the loss
class_weights = torch.tensor(
    compute_class_weight("balanced", classes=np.unique(train_df["label"]), y=train_df["label"]),
    dtype=torch.float,
)
print("class weights:", dict(zip(LABELS.values(), class_weights.round(decimals=3).tolist())))

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=class_weights.to(outputs.logits.device))(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
import json
from sklearn.metrics import classification_report, confusion_matrix

# Best epoch by macro F1 (load_best_model_at_end)
preds = np.argmax(trainer.predict(test_ds).predictions, axis=-1)
y_true = np.array(test_ds["labels"])

report = classification_report(y_true, preds, target_names=list(LABELS.values()), digits=3)
print(report)
print("Confusion matrix (rows = true, cols = predicted):")
print(confusion_matrix(y_true, preds))

# Save to Drive: the model files plus the scores, so the team can check a local copy reproduces them
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
with open(f"{SAVE_DIR}/metrics.json", "w") as f:
    json.dump({
        "accuracy": accuracy_score(y_true, preds),
        "f1_macro": f1_score(y_true, preds, average="macro"),
        "report": classification_report(y_true, preds, target_names=list(LABELS.values()), output_dict=True),
        "train_rows": len(train_df),
        "test_rows": len(test_df),
    }, f, indent=2)

!ls -la "{SAVE_DIR}"

In [ ]:
# Sanity check on a few made-up posts
examples = [
    "Just got my second dose, grateful to the nurses who made it so easy",
    "Not letting them inject my kids with that poison",
    "Does anyone know if the clinic on Main St takes walk-ins for boosters?",
]
model.eval()
inputs = tokenizer([clean_text(t) for t in examples], return_tensors="pt", padding=True, truncation=True).to(model.device)
with torch.no_grad():
    probs = model(**inputs).logits.softmax(-1)
for text, p in zip(examples, probs):
    print(f"{LABELS[int(p.argmax())]:>8} {p.max():.2f}  {text}")